# BIG DATA: PROCESSAMENTO COM LAZY EVALUATION (PLANO DE EXECUÇÃO) E ANÁLISE ESTATÍSTICA


## Situação de Aprendizagem — Trabalhando "Na Ponta" sem Estourar a Memória (*Data Organized*)

Na Aula 12, concluímos a etapa de **Pré-processamento**: pegamos 3 meses de arquivos `.csv` brutos do Novo Bolsa Família (~200 MB), corrigimos os tipos das colunas e compactamos tudo em um único arquivo: **`bolsa_familia_3meses.parquet`** (**2.886.000 registros**).

Hoje entramos na **Etapa 2 do Pipeline de Big Data: o Processamento e a Análise Estatística**. Na reunião técnica da **Data Organized**, o Analista Sênior explicou o grande desafio de quem trabalha **"na ponta"** (ou seja, o Cientista/Analista de Dados rodando análises no próprio notebook ou desktop, sem um cluster de 50 servidores na nuvem):

"Mesmo usando arquivos `.parquet`, se você carregar tabelas de vários gigabytes inteiras na memória RAM antes de filtrar o que precisa, o seu computador vai engasgar. Hoje vocês vão aprender a usar o cérebro otimizador do Polars: a **Avaliação Preguiçosa (*Lazy Evaluation*)**. Em vez de sair executando cada linha de código na hora, o Polars monta primeiro um **Plano de Execução Otimizado** (`.lazy()` e `.explain()`), elimina tudo o que é inútil antes mesmo de ler o disco e só processa os dados quando você aperta o gatilho (`.collect()`)."

### Bibliotecas Utilizadas

*Instalação (caso ainda não estejam instaladas no seu ambiente virtual):*

```bash
pip install polars numpy pandas psutil
```

## Bloco 1: Modo Ansioso (*Eager*) vs. Modo Preguiçoso (*Lazy Evaluation*)

### Analogia do Dia a Dia: O Comprador Impulsivo vs. O Comprador Planejador
Imagine que você recebeu uma lista de compras com 4 instruções para fazer no supermercado:
1. *Pegar todas as frutas da banca.*
2. *Calcular o preço com desconto de cada fruta.*
3. *Jogar fora todas as frutas, menos as maçãs.*
4. *Mostrar apenas o nome e o preço final das maçãs.*

Veja como dois compradores diferentes resolvem essa tarefa:

- **Comprador 1 — Modo Ansioso / Imediato (*Eager Evaluation* — Pandas e `pl.read_parquet`):**
  - Ele obedece cada linha **na hora em que lê**! Enche o carrinho com 50 tipos de frutas (gastando espaço), calcula o desconto de todas as 50 (gastando tempo), e só no passo 3 joga 49 fora para ficar apenas com as maçãs.
  - **No computador:** Cada linha de código cria uma cópia intermediária na memória RAM antes de saber qual será o próximo comando.

- **Comprador 2 — Modo Preguiçoso / Planejado (*Lazy Evaluation* — `.lazy()` e `pl.scan_parquet`):**
  - Ele **não pega nada** enquanto lê a lista! Ele lê a lista inteira até o final, pensa e percebe: *"Ora, se no final eu só quero o nome e o preço das maçãs, eu vou direto na caixa de maçãs e ignoro todo o resto do mercado!"*
  - **No computador:** O Polars monta uma **receita otimizada** na memória e executa tudo em uma única passada ultrarrápida.

### Os 3 Pilares do Ciclo de Vida Lazy no Polars:
| Etapa | Comando Polars | O que acontece dentro da máquina? |
| :--- | :--- | :--- |
| **1. Planejar** | `.lazy()` (na RAM) ou `pl.scan_parquet()` (no disco) | Cria um `LazyFrame`. Nenhum dado é carregado ou calculado ainda; apenas registra as intenções. |
| **2. Inspecionar** | `.explain()` | Exibe o **Plano de Execução Otimizado** em texto, mostrando como o Polars reordenou seus comandos para economizar RAM e CPU. |
| **3. Executar** | `.collect()` | Dispara o motor em Rust usando todos os núcleos da CPU e entrega o `DataFrame` final pronto na memória. |

## Bloco 2: Por Dentro do Otimizador de Consultas (*Query Optimizer*)

Quando você usa `.lazy()` ou `pl.scan_parquet()`, o motor do Polars aplica automaticamente **4 técnicas avançadas de otimização** antes de tocar nos dados:

1. **Empurrar Filtros para Baixo (*Predicate Pushdown*):**
   - Mesmo que você escreva `.filter(pl.col('UF') == 'RJ')` lá no final do seu código, o Polars move esse filtro para a **primeira etapa da leitura do arquivo Parquet**. Linhas de outros estados nem sequer são carregadas na memória RAM!
2. **Empurrar Seleção de Colunas (*Projection Pushdown*):**
   - O otimizador analisa todas as colunas usadas nas suas contas e descarta automaticamente as colunas que não aparecem no resultado final. Se o arquivo tem 8 colunas e você só usa 2, ele lê apenas `2/8` colunas do SSD.
3. **Corte Antecipado de Linhas (*Slice Pushdown*):**
   - Se você pedir `.head(10)` no final de uma consulta simples, o Polars para de ler o arquivo assim que encontra as 10 primeiras linhas válidas, sem precisar varrer milhões de registros.
4. **Execução em Streaming (`collect(engine='streaming')`):**
   - Se um arquivo Parquet tiver **30 GB** e o seu computador tiver apenas **8 GB de RAM**, o modo *streaming* fatia o arquivo em pequenos lotes, processa um lote por vez e junta o resultado final sem nunca estourar a memória!

## Bloco 3: Como Ler o Plano de Execução (`.explain()`) sem Confusão

Quando rodamos `print(plano.explain())`, o Polars mostra uma árvore de texto. **Regra de leitura: leia sempre de BAIXO para CIMA** (a base é a leitura do arquivo no disco; o topo é o resultado final que aparece na sua tela).

### Tabela de Tradução dos Termos do `.explain()`:
| Termo que aparece no `.explain()` | Tradução para o Português Claro |
| :--- | :--- |
| **`Parquet SCAN [arquivo.parquet]`** | Passo inicial: abrindo o arquivo Parquet diretamente no SSD. |
| **`PROJECT 2/8 COLUMNS`** | *Projection Pushdown* funcionando! Ele ignorou 6 colunas inúteis e leu só 2. |
| **`SELECTION: [(col("UF")) == ("RJ")]`** | *Predicate Pushdown* funcionando! Ele filtrou o estado RJ direto na leitura do disco. |
| **`AGGREGATE [mean, median, sum]`** | Calculando todas as métricas estatísticas em paralelo nos núcleos da CPU. |
| **`SORT BY [col("Total")]`** | Ordenando a tabela final resumida (que já está pequena). |

## Bloco 4: Dicionário de Dados da Base (`bolsa_familia_3meses.parquet`)

Vamos trabalhar sobre o arquivo `.parquet` consolidado na Aula 12 (**2.886.000 registros** e **8 colunas** já tipadas):

| # | Nome da Coluna | Tipo no Parquet | Descrição e Papel na Análise Estatística |
| :-: | :--- | :--- | :--- |
| **1** | `MES_COMPETENCIA` | `String` | Ano e mês de competência da folha (`'202601'`, `'202602'`, `'202603'`). Variável qualitativa ordinal. |
| **2** | `MES_REFERENCIA` | `String` | Ano e mês de referência do pagamento (`'202601'`, `'202602'`, `'202603'`). |
| **3** | `UF` | `String` | Sigla do Estado beneficiário (`'RJ'`, `'SP'`, `'MG'`, `'BA'`, `'PE'`, `'CE'`, `'PA'`, `'MA'`, `'PR'`, `'RS'`). |
| **4** | `CODIGO_MUNICIPIO_SIAFI` | `Int64` | Código numérico oficial do município no sistema federal. |
| **5** | `NOME_MUNICIPIO` | `String` | Nome oficial do município em letras maiúsculas (`'RIO DE JANEIRO'`, `'SAO PAULO'`, etc.). |
| **6** | `NIS_FAVORECIDO` | `Int64` | Número de Identificação Social (11 dígitos) do titular do benefício. |
| **7** | `NOME_FAVORECIDO` | `String` | Nome completo do responsável familiar titular. |
| **8** | `VALOR_PARCELA` | `Float64` | Valor pago em Reais (já limpo e convertido para número decimal na Aula 12). Variável quantitativa contínua. |

## Exemplo 1 (Nível Básico): Vendo o Otimizador Reorganizar seu Código (`.lazy()`, `.explain()`, `.collect()`)

Vamos criar uma pequena tabela na memória e escrever um código "desorganizado" de propósito (primeiro calculando colunas novas para todo mundo, e só no final filtrando um estado e escolhendo colunas) para ver como o `.explain()` corrige a ordem sozinho!

In [3]:
# Importa a biblioteca Polars com o apelido padrão 'pl'
import polars as pl

# Cria um DataFrame pequeno na memória com 6 registros simulados
df_exemplo = pl.DataFrame({
    'UF': ['RJ', 'SP', 'BA', 'RJ', 'MG', 'SP'],                                                    # Coluna de estados
    'MUNICIPIO': ['NITEROI', 'CAMPINAS', 'SALVADOR', 'DUQUE DE CAXIAS', 'BELO HORIZONTE', 'SANTOS'], # Coluna de municípios
    'NIS': [1001, 1002, 1003, 1004, 1005, 1006],                                                   # Coluna que não será usada no final
    'VALOR_PARCELA': [600.0, 850.0, 720.0, 950.0, 680.0, 1100.0]                                   # Valor mensal pago
})

# Constrói o plano preguiçoso (.lazy()) com filtro escrito propositalmente no final
plano_basico = (
    df_exemplo
    .lazy()                                                                 # 1. Ativa o modo Lazy (não calcula nada ainda!)
    .with_columns((pl.col('VALOR_PARCELA') * 12).alias('TOTAL_ANUAL'))      # 2. Planeja multiplicar cada parcela por 12 meses
    .filter(pl.col('UF') == 'BA')                                           # 3. Planeja filtrar apenas o estado 'RJ'
    .select(['MUNICIPIO', 'VALOR_PARCELA', 'TOTAL_ANUAL'])                  # 4. Planeja manter apenas estas 3 colunas
)

# Verifica que o objeto criado ainda é apenas um plano (LazyFrame), não uma tabela pronta
print("Tipo do objeto antes de executar:", type(plano_basico))

# Imprime o Plano de Execução Otimizado gerado pelo Polars (leia de baixo para cima!)
print("\n=== PLANO DE EXECUÇÃO OTIMIZADO (.explain()) ===")
print(plano_basico.explain())

Tipo do objeto antes de executar: <class 'polars.lazyframe.frame.LazyFrame'>

=== PLANO DE EXECUÇÃO OTIMIZADO (.explain()) ===
 WITH_COLUMNS:
 [(col("VALOR_PARCELA") * 12.0).alias("TOTAL_ANUAL")] 
  simple π 2/2 ["MUNICIPIO", "VALOR_PARCELA"]
    FILTER col("UF") == "BA"
    FROM
      DF ["UF", "MUNICIPIO", "NIS", "VALOR_PARCELA"]; PROJECT["MUNICIPIO", "VALOR_PARCELA", "UF"] 3/4 COLUMNS


In [4]:
# Agora chamamos .collect() para disparar a execução real do plano otimizado
resultado_final = plano_basico.collect()

# Exibe a tabela final materializada na memória
print("=== TABELA FINAL APÓS .collect() ===")
print(resultado_final)

=== TABELA FINAL APÓS .collect() ===
shape: (1, 3)
┌───────────┬───────────────┬─────────────┐
│ MUNICIPIO ┆ VALOR_PARCELA ┆ TOTAL_ANUAL │
│ ---       ┆ ---           ┆ ---         │
│ str       ┆ f64           ┆ f64         │
╞═══════════╪═══════════════╪═════════════╡
│ SALVADOR  ┆ 720.0         ┆ 8640.0      │
└───────────┴───────────────┴─────────────┘


*Leitura Analítica (Exemplo 1):*

Leia o `.explain()` de baixo para cima: o Polars percebeu que a coluna `NIS` nunca é usada (`PROJECT 3/4 COLUMNS`) e **antecipou o filtro `UF == 'RJ'`** (`FILTER`) antes de multiplicar os valores por 12! Em vez de calcular 6 linhas, ele calculou apenas as 2 linhas do Rio de Janeiro.

## Exemplo 2 (Nível Básico-Intermediário): Preparando o Arquivo Parquet e Medindo CPU/RAM

Antes de compararmos a execução ansiosa (*Eager*) com a execução preguiçosa (*Lazy*) em escala de milhões de linhas, vamos carregar nossas ferramentas de monitoramento de hardware (`psutil`, `time`, `gc`) e garantir que o arquivo **`bolsa_familia_3meses.parquet`** (**2.886.000 registros**) da Aula 12 esteja pronto no disco.

In [5]:
# Importa 'os' para verificar arquivos no disco e 'time' para cronometrar segundos
import os
import time

# Importa 'gc' (Garbage Collector) para limpar memória RAM entre os testes
import gc

# Importa 'psutil' para medir exatamente quanta RAM e quantos núcleos de CPU temos na máquina
import psutil

# Importa NumPy caso seja necessário gerar o arquivo Parquet de contingência
import numpy as np

# Define uma função que lê no sistema operacional a RAM física usada pelo Python em MB
def medir_ram_mb():
    processo_atual = psutil.Process(os.getpid())             # Obtém o identificador do processo Python
    return processo_atual.memory_info().rss / (1024 ** 2)    # Converte bytes para Megabytes (MB)

# Define o caminho do arquivo Parquet consolidado na Aula 12
caminho_parquet = 'bolsa_familia_3meses.parquet'

# Garante que o arquivo exista na pasta (se não existir, gera os 2.886.000 registros agora)
if not os.path.exists(caminho_parquet):
    print("Gerando 'bolsa_familia_3meses.parquet' (2.886.000 registros)...")
    np.random.seed(42)
    n_total = 2_886_000
    pl.DataFrame({
        'MES_COMPETENCIA': np.repeat(['202601', '202602', '202603'], 962_000),
        'MES_REFERENCIA': np.repeat(['202601', '202602', '202603'], 962_000),
        'UF': np.random.choice(['RJ', 'SP', 'MG', 'BA', 'PE', 'CE', 'PA', 'MA', 'PR', 'RS'], size=n_total),
        'CODIGO_MUNICIPIO_SIAFI': np.random.randint(1000, 9999, size=n_total, dtype=np.int64),
        'NOME_MUNICIPIO': np.random.choice(['RIO DE JANEIRO', 'SAO PAULO', 'BELO HORIZONTE', 'SALVADOR', 'FORTALEZA', 'RECIFE', 'CURITIBA', 'PORTO ALEGRE'], size=n_total),
        'NIS_FAVORECIDO': np.random.randint(10000000000, 99999999999, size=n_total, dtype=np.int64),
        'NOME_FAVORECIDO': np.random.choice(['MARIA DA SILVA SANTOS', 'JOSE OLIVEIRA SOUZA', 'ANA PAULA FERREIRA', 'CARLOS EDUARDO LIMA', 'FRANCISCA DAS CHAGAS'], size=n_total),
        'VALOR_PARCELA': np.round(np.random.uniform(600.0, 1150.0, size=n_total), 2)
    }).write_parquet(caminho_parquet)

# Exibe o diagnóstico de hardware da estação de trabalho e do arquivo Parquet
print(f"Núcleos lógicos de CPU disponíveis para o Polars: {psutil.cpu_count(logical=True)} núcleos")
print(f"Arquivo Parquet pronto: {caminho_parquet} ({os.path.getsize(caminho_parquet)/(1024**2):.2f} MB no disco)")
print(f"RAM inicial ocupada pelo Python: {medir_ram_mb():.1f} MB")

Gerando 'bolsa_familia_3meses.parquet' (2.886.000 registros)...
Núcleos lógicos de CPU disponíveis para o Polars: 20 núcleos
Arquivo Parquet pronto: bolsa_familia_3meses.parquet (32.19 MB no disco)
RAM inicial ocupada pelo Python: 542.0 MB


*Leitura Analítica (Exemplo 2):*

Inspecionar os recursos físicos da máquina (`psutil.cpu_count()` e consumo de RAM) antes de processar grandes bases é uma diretriz explícita do nosso Plano de Curso para trabalhar com dados em larga escala na ponta.

## Exemplo 3 (Nível Intermediário): Duelo de Performance e RAM — Sem Lazy (`read_parquet`) vs. Com Lazy (`scan_parquet`)

Vamos executar a mesma tarefa estatística sobre os **2.886.000 registros** de duas formas diferentes e comparar o tempo e a memória RAM gasta:
- **Forma A (Sem Lazy / Eager — `pl.read_parquet`):** Carrega todas as 8 colunas e 2,88 milhões de linhas na RAM primeiro, e só depois filtra o estado `'RJ'` no mês `'202603'`.
- **Forma B (Com Lazy — `pl.scan_parquet` + `.collect()`):** Empurra o filtro e a seleção de colunas direto para a leitura do disco, trazendo apenas os dados filtrados para a RAM.

In [6]:
# ==============================================================================
# TESTE A: EXECUÇÃO IMEDIATA / EAGER (pl.read_parquet)
# ==============================================================================
gc.collect()                                                                 # Limpa lixo de memória antes do teste
ram_antes_eager = medir_ram_mb()                                             # Registra RAM inicial
t0_eager = time.time()                                                       # Inicia cronômetro

# Lê o arquivo Parquet inteiro (todas as 8 colunas e 2.886.000 linhas) para a RAM
df_eager = pl.read_parquet(caminho_parquet)
ram_pico_eager = medir_ram_mb() - ram_antes_eager                            # Mede quanto a RAM subiu ao abrir tudo

# Só depois de carregar tudo na RAM é que aplica o filtro e calcula as métricas
res_eager = (
    df_eager
    .filter((pl.col('UF') == 'RJ') & (pl.col('MES_COMPETENCIA') == '202603'))
    .select([
        pl.col('VALOR_PARCELA').count().alias('Qtd_Familias'),
        pl.col('VALOR_PARCELA').mean().alias('Media_BRL'),
        pl.col('VALOR_PARCELA').median().alias('Mediana_BRL'),
        pl.col('VALOR_PARCELA').sum().alias('Total_Pago_BRL')
    ])
)
tempo_eager = time.time() - t0_eager                                         # Calcula tempo total do modo Eager

# Apaga o DataFrame Eager da memória RAM antes de iniciar o teste Lazy
del df_eager, res_eager
gc.collect()

# ==============================================================================
# TESTE B: EXECUÇÃO PREGUIÇOSA / LAZY (pl.scan_parquet + .collect())
# ==============================================================================
ram_antes_lazy = medir_ram_mb()                                              # Registra RAM inicial
t0_lazy = time.time()                                                        # Inicia cronômetro

# Monta o plano otimizado diretamente sobre o arquivo no disco (0 MB carregados ainda)
plano_lazy = (
    pl.scan_parquet(caminho_parquet)
    .filter((pl.col('UF') == 'RJ') & (pl.col('MES_COMPETENCIA') == '202603'))
    .select([
        pl.col('VALOR_PARCELA').count().alias('Qtd_Familias'),
        pl.col('VALOR_PARCELA').mean().alias('Media_BRL'),
        pl.col('VALOR_PARCELA').median().alias('Mediana_BRL'),
        pl.col('VALOR_PARCELA').sum().alias('Total_Pago_BRL')
    ])
)

# Executa o plano já filtrando e podando colunas durante a leitura do disco
res_lazy = plano_lazy.collect()
tempo_lazy = time.time() - t0_lazy                                           # Calcula tempo total do modo Lazy
ram_pico_lazy = max(0.5, medir_ram_mb() - ram_antes_lazy)                    # Mede o aumento mínimo de RAM

# Exibe o comparativo lado a lado
print(f"[SEM LAZY - Eager] Tempo: {tempo_eager:.4f}s | Pico de RAM: +{ram_pico_eager:.1f} MB")
print(f"[COM LAZY - Scan]  Tempo: {tempo_lazy:.4f}s | Pico de RAM: +{ram_pico_lazy:.1f} MB")
print("-" * 68)
print(res_lazy)

[SEM LAZY - Eager] Tempo: 0.1225s | Pico de RAM: +-7.1 MB
[COM LAZY - Scan]  Tempo: 0.0105s | Pico de RAM: +2.9 MB
--------------------------------------------------------------------
shape: (1, 4)
┌──────────────┬────────────┬─────────────┬────────────────┐
│ Qtd_Familias ┆ Media_BRL  ┆ Mediana_BRL ┆ Total_Pago_BRL │
│ ---          ┆ ---        ┆ ---         ┆ ---            │
│ u32          ┆ f64        ┆ f64         ┆ f64            │
╞══════════════╪════════════╪═════════════╪════════════════╡
│ 96068        ┆ 875.568722 ┆ 876.06      ┆ 8.4114e7       │
└──────────────┴────────────┴─────────────┴────────────────┘


*Leitura Analítica (Exemplo 3):*

Este é o segredo para processar **14 GB ou 30 GB de dados** em um computador comum: no modo **Eager**, o Python precisa alocar memória para a tabela inteira antes de filtrar; no modo **Lazy (`scan_parquet`)**, ele filtra as linhas e descarta as colunas desnecessárias **enquanto lê do SSD**, mantendo o consumo de RAM quase flat!

## Exemplo 4 (Nível Intermediário-Avançado): Análise Estatística Descritiva Completa em Larga Escala

Agora vamos realizar um **estudo estatístico completo da distribuição dos pagamentos por Estado (`UF`)** sobre os **2.886.000 registros**, calculando em um único pipeline otimizado:
- **Posição e Centro:** Contagem, Média, Mediana ($Q2$), 1º Quartil ($Q1$), 3º Quartil ($Q3$) e Distância Média-Mediana ($\%$).
- **Dispersão:** Intervalo Interquartil ($IQR = Q3 - Q1$), Desvio Padrão (`ddof=0`) e Coeficiente de Variação ($CV\%$).
- **Forma:** Assimetria (`.skew()`).

In [8]:
# Constrói o plano de análise estatística completa usando scan_parquet()
plano_auditoria_uf = (
    pl.scan_parquet(caminho_parquet)                                         # Abre leitura preguiçosa do Parquet
    .group_by('UF')                                                          # Agrupa os 2,88M registros por Estado
    .agg([
        pl.col('VALOR_PARCELA').count().alias('Qtd_Beneficios'),             # Total de pagamentos
        pl.col('VALOR_PARCELA').mean().alias('Media_BRL'),                   # Média aritmética
        pl.col('VALOR_PARCELA').median().alias('Mediana_BRL'),               # Mediana (50%)
        pl.col('VALOR_PARCELA').quantile(0.25).alias('Q1_BRL'),              # 1º Quartil (25%)
        pl.col('VALOR_PARCELA').quantile(0.75).alias('Q3_BRL'),              # 3º Quartil (75%)
        pl.col('VALOR_PARCELA').std(ddof=0).alias('Desvio_Padrao'),          # Desvio padrão populacional
        pl.col('VALOR_PARCELA').skew().alias('Assimetria')                   # Assimetria da distribuição
    ])
    # Calcula métricas relativas em paralelo a partir das colunas agregadas acima
    .with_columns([
        (pl.col('Q3_BRL') - pl.col('Q1_BRL')).alias('IQR_BRL'),              # Intervalo Interquartil (Q3 - Q1)
        ((pl.col('Desvio_Padrao') / pl.col('Media_BRL')) * 100).alias('CV_pct'), # Coeficiente de Variação (%)
        (((pl.col('Media_BRL') - pl.col('Mediana_BRL')) / pl.col('Mediana_BRL')) * 100).alias('Dist_MM_pct')
    ])
    .sort('Qtd_Beneficios', descending=True)                                 # Ordena do maior estado para o menor
)

# 1. Mostra o Plano de Execução Otimizado antes de rodar
print("=== PLANO DE EXECUÇÃO (.explain()) ===")
print(plano_auditoria_uf.explain())

# 2. Executa o plano e cronometra
t0_aud = time.time()
df_estatistica_uf = plano_auditoria_uf.collect()
tempo_aud = time.time() - t0_aud

print(f"\n=== DIAGNÓSTICO ESTATÍSTICO POR ESTADO (Executado em {tempo_aud:.4f}s) ===")
print(df_estatistica_uf)

=== PLANO DE EXECUÇÃO (.explain()) ===
SORT BY [descending: [true]] [col("Qtd_Beneficios")]
   WITH_COLUMNS:
   [(col("Q3_BRL") - col("Q1_BRL")).alias("IQR_BRL"), ((col("Desvio_Padrao") / col("Media_BRL")) * 100.0).alias("CV_pct"), (((col("Media_BRL") - col("Mediana_BRL")) / col("Mediana_BRL")) * 100.0).alias("Dist_MM_pct")] 
    AGGREGATE[maintain_order: false]
      [col("VALOR_PARCELA").count().alias("Qtd_Beneficios"), col("VALOR_PARCELA").mean().alias("Media_BRL"), col("VALOR_PARCELA").median().alias("Mediana_BRL"), col("VALOR_PARCELA").quantile([dyn float: 0.25]).alias("Q1_BRL"), col("VALOR_PARCELA").quantile([dyn float: 0.75]).alias("Q3_BRL"), col("VALOR_PARCELA").std().alias("Desvio_Padrao"), col("VALOR_PARCELA").skew().alias("Assimetria")] BY [col("UF")]
      FROM
      Parquet SCAN [bolsa_familia_3meses.parquet]
      PROJECT 2/8 COLUMNS
      ESTIMATED ROWS: 2886000

=== DIAGNÓSTICO ESTATÍSTICO POR ESTADO (Executado em 0.1129s) ===
shape: (10, 11)
┌─────┬─────────────┬──────

*Leitura Analítica (Exemplo 4 — Inferência Estatística sobre Big Data):*

Observe como todas as técnicas do curso se unem aqui:
1. No `.explain()`, o Polars leu apenas **2 de 8 colunas** (`PROJECT 2/8 COLUMNS`: `UF` e `VALOR_PARCELA`).
2. Em todos os estados, a **Média (`~R$ 875,00`)** e a **Mediana (`~R$ 875,00`)** coincidem (`Dist_MM_pct ≈ 0,0%`), a **Assimetria** fica em torno de `0,00` (distribuição **Simétrica**) e o **Coeficiente de Variação (`CV_pct ≈ 18,1%`)** confirma estabilidade homogênea entre os estados.

---
## Atividades Práticas (Do Básico ao Intermediário)

### ATIVIDADE 1 (Nível Básico) — Montando e Inspecionando um Plano Lazy (`.lazy()` e `.explain()`)
Crie um `pl.DataFrame` representando 6 pedidos de clientes corporativos da **Data Organized**:
```python
df_contratos = pl.DataFrame({
    'Cliente': ['Alfa', 'Beta', 'Gama', 'Delta', 'Epsilon', 'Zeta'],
    'Setor': ['Varejo', 'Saude', 'Varejo', 'Educacao', 'Varejo', 'Saude'],
    'Coluna_Inutil': [999, 888, 777, 666, 555, 444],
    'Valor_Mensal': [2000.0, 4500.0, 1500.0, 3000.0, 5000.0, 3800.0]
})
```

**Faça:**
1. Transforme `df_contratos` em um plano preguiçoso usando `.lazy()`.
2. Encadeie: (a) criar a coluna `Valor_Semestral` (`Valor_Mensal * 6`), (b) filtrar apenas o setor `'Varejo'` e (c) selecionar apenas `['Cliente', 'Valor_Mensal', 'Valor_Semestral']`.
3. Exiba o `.explain()` para comprovar que `Coluna_Inutil` foi descartada antes do cálculo e depois execute com `.collect()`.

In [9]:
# Escreva aqui o código da Atividade 1 (Nível Básico):

# Cria DataFrame de contratos
df_contratos = pl.DataFrame({
    'Cliente': ['Alfa', 'Beta', 'Gama', 'Delta', 'Epsilon', 'Zeta'],
    'Setor': ['Varejo', 'Saude', 'Varejo', 'Educacao', 'Varejo', 'Saude'],
    'Coluna_Inutil': [999, 888, 777, 666, 555, 444],
    'Valor_Mensal': [2000.0, 4500.0, 1500.0, 3000.0, 5000.0, 3800.0]
})

# Cria plano de execução 'lazy' (atividade 1.1) para criar coluna 'Valor_Semestral', filtrar o setor de Varejo e retornar apenas as colunas 'Cliente', 'Valor_Mensal' e 'Valor_Semestral' (atividade 1.2)
plano_exec = (df_contratos
    .lazy()
    .with_columns((pl.col('Valor_Mensal') * 6).alias('Valor_Semestral')) 
    .filter(pl.col('Setor') == 'Varejo')
    .select(['Cliente', 'Valor_Mensal', 'Valor_Semestral'])
    .sort('Valor_Mensal', descending=True)                                 # Ordena do maior estado para o menor

)

# Exibe o plano de execução e executa a consulta (atividade 1.3)
print("Plano de Execução:")
print(plano_exec.explain())
df_result = plano_exec.collect()
print(df_result)

Plano de Execução:
SORT BY [descending: [true]] [col("Valor_Mensal")]
   WITH_COLUMNS:
   [(col("Valor_Mensal") * 6.0).alias("Valor_Semestral")] 
    simple π 2/2 ["Cliente", "Valor_Mensal"]
      FILTER col("Setor") == "Varejo"
      FROM
        DF ["Cliente", "Setor", "Coluna_Inutil", "Valor_Mensal"]; PROJECT["Cliente", "Valor_Mensal", "Setor"] 3/4 COLUMNS
shape: (3, 3)
┌─────────┬──────────────┬─────────────────┐
│ Cliente ┆ Valor_Mensal ┆ Valor_Semestral │
│ ---     ┆ ---          ┆ ---             │
│ str     ┆ f64          ┆ f64             │
╞═════════╪══════════════╪═════════════════╡
│ Epsilon ┆ 5000.0       ┆ 30000.0         │
│ Alfa    ┆ 2000.0       ┆ 12000.0         │
│ Gama    ┆ 1500.0       ┆ 9000.0          │
└─────────┴──────────────┴─────────────────┘


### ATIVIDADE 2 (Nível Intermediário) — Estudo Estatístico Municipal sobre Big Data (`scan_parquet`)
Utilizando o arquivo `'bolsa_familia_3meses.parquet'` (**2.886.000 registros**), realize um estudo comparativo da distribuição dos benefícios entre os municípios de `'RIO DE JANEIRO'`, `'SAO PAULO'`, `'SALVADOR'` e `'FORTALEZA'` no mês `'202603'`:

**Faça:**
1. Abra o arquivo em modo preguiçoso com `pl.scan_parquet('bolsa_familia_3meses.parquet')`.
2. Filtre apenas os 4 municípios listados (`pl.col('NOME_MUNICIPIO').is_in([...])`) e `MES_COMPETENCIA == '202603'`.
3. Agrupe por `NOME_MUNICIPIO` e calcule:
   - `Qtd_Familias` (`count`)
   - `Total_Pago_BRL` (`sum`)
   - `Media_BRL` (`mean`) e `Mediana_BRL` (`median`)
   - `Desvio_Padrao` (`std(ddof=0)`) e `Assimetria` (`skew`)
4. Crie a coluna derivada `CV_pct` (`(Desvio_Padrao / Media_BRL) * 100`) com `.with_columns()`, imprima o `.explain()`, execute o `.collect()` e responda: qual município recebeu o maior repasse total e como se comporta a simetria da distribuição?

In [ ]:
# Escreva aqui o código da Atividade 2 (Nível Intermediário):

# Abre o arquivo parquet do bolsa família 
from pathlib import Path

arq_parquet = Path('..') / 'DATA' / 'bolsa_familia_3meses.parquet'
plano_exec = (
    pl.scan_parquet(arq_parquet)                                        # Abre leitura preguiçosa do Parquet (atividade 2.1)
    .filter(pl.col('NOME_MUNICIPIO').is_in(['RIO DE JANEIRO', 'SAO PAULO', 'SALVADOR', 'FORTALEZA']), pl.col('MES_REFERENCIA') == 202603) # Filtra os municípios e competência solicitados (atividade 2.2)
    .group_by('NOME_MUNICIPIO')                                         # Agrupa por município s e criar colunas agregadoras abaixo (atividade 2.3)
    .agg([
        pl.col('VALOR_PARCELA').count().alias('Qtd_Familia'),
        pl.col('VALOR_PARCELA').sum().alias('Total_Pago_BRL'),
        pl.col('VALOR_PARCELA').median().alias('Mediana_BRL'),
        pl.col('VALOR_PARCELA').mean().alias('Media_BRL'),
        pl.col('VALOR_PARCELA').std(ddof=0).alias('Desvio_Padrao'), 
        pl.col('VALOR_PARCELA').skew().alias('Assimetria')
    ])
    .with_columns([
        ((pl.col('Desvio_Padrao') / pl.col('Media_BRL')) * 100).alias('CV_pct')
    ])
    .sort('Total_Pago_BRL', descending=True)
)

# Mostra o Plano de Execução Otimizado e executa a consulta (atividade 2.4)
print('Plano de Execução:')
print(plano_exec.explain())
df_result = plano_exec.collect()
print(df_result.head())


Plano de Execução:
SORT BY [descending: [true]] [col("Total_Pago_BRL")]
   WITH_COLUMNS:
   [((col("Desvio_Padrao") / col("Media_BRL")) * 100.0).alias("CV_pct")] 
    AGGREGATE[maintain_order: false]
      [col("VALOR_PARCELA").count().alias("Qtd_Familia"), col("VALOR_PARCELA").sum().alias("Total_Pago_BRL"), col("VALOR_PARCELA").median().alias("Mediana_BRL"), col("VALOR_PARCELA").mean().alias("Media_BRL"), col("VALOR_PARCELA").std().alias("Desvio_Padrao"), col("VALOR_PARCELA").skew().alias("Assimetria")] BY [col("NOME_MUNICIPIO")]
      FROM
      simple π 2/2 ["NOME_MUNICIPIO", ... 1 other column]
        Parquet SCAN [../DATA/bolsa_familia_3meses.parquet]
        PROJECT 3/7 COLUMNS
        SELECTION: (col("MES_REFERENCIA") == 202603) & col("NOME_MUNICIPIO").is_in([["RIO DE JANEIRO", "SAO PAULO", … "FORTALEZA"]])
        ESTIMATED ROWS: 2886000
shape: (4, 8)
┌────────────┬────────────┬────────────┬───────────┬───────────┬───────────┬───────────┬───────────┐
│ NOME_MUNIC ┆ Qtd_Famili 

*Leitura Analítica Final:*

Com o **Lazy Evaluation (`scan_parquet` + `explain` + `collect`)**, você agora consegue executar auditorias estatísticas completas sobre milhões de registros lendo apenas as colunas necessárias (`3/8 colunas`) e filtrando os dados diretamente na leitura do disco. Na **Aula 14**, vamos aprender a otimizar ainda mais o processamento de colunas de texto repetidas usando **Memória Cache (`pl.StringCache()`)**, além de calcular **Frequência, Moda e Histogramas** em Big Data!